In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
len(documents)

72

In [4]:
from minsearch import Index

In [5]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [6]:
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

In [7]:
index = build_index(documents)

In [8]:
index.search(
    "How does the agentic loop keep calling the model until it stops?",
    num_results=1
)[0].get("filename")

'01-agentic-rag/lessons/14-agentic-loop.md'

In [9]:
from rag_helper import RAGBase

In [10]:
from dotenv import load_dotenv
load_dotenv()
import os

from openai import OpenAI
# openai_client = OpenAI()
llm_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [11]:
assistant = RAGBase(index, llm_client, model="llama-3.3-70b-versatile")

In [12]:
resp = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [15]:
resp.usage.input_tokens

7088

In [16]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [17]:
len(chunks)

295

In [18]:
chunk_index = build_index(chunks)

In [19]:
assistant2 = RAGBase(chunk_index, llm_client, model="llama-3.3-70b-versatile")

In [20]:
resp = assistant2.rag("How does the agentic loop keep calling the model until it stops?")
resp.usage.input_tokens

2217

In [21]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""
question = """
How does the agentic loop work, and how is it different from plain RAG?
"""

In [25]:
import json

In [38]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search_chunks':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [39]:
def search_chunks(query):
    return chunk_index.search(
        query,
        num_results=5,
    )

In [40]:
search_tool = {
    "type": "function",
    'name': 'search_chunks',
    'description': 'Search the course lessons modules divided by chunks for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course lessons modules.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [45]:
def agent_loop(instructions, question, model='llama-3.3-70b-versatile') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = llm_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output[1:])

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [46]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search_chunks {"query":"agentic loop vs plain RAG"}
iteration #2...
function_call: search_chunks {"query":"agentic loop vs RAG"}
iteration #3...
ASSISTANT:
The agentic loop is a loop that keeps calling the model until it returns a response without any function calls. The model reasons about the next action, and the code performs it, and the model sees the result on the next turn. The loop stops when the model returns a final answer with no more tool calls.

The agentic loop is different from plain RAG (Retrieve, Augment, Generate) in that it allows the model to decide what to do at each step, rather than following a fixed flow. The model can search again with different terms, ask the user a clarifying question, or fix typos. The agentic loop is more flexible and can handle more complex scenarios than plain RAG.

In the context of the provided text, the agentic loop is used to build an agent that can answer questions from a course student. The agent uses a